执行命令需要在同一行命令中，先 source 环境名（base、modelone 等）才能 pip 安装到指定环境；如果不知道有哪些虚拟环境，可以运行 conda info --envs 查看

In [ ]:
!source activate modelone && pip install pandas

In [ ]:
import json, os, time, shutil, pandas

from cubestudio.request.model_client import Client,init
from cubestudio.dataset.dataset import Dataset
from pandarallel import pandarallel

# Initialization
pandarallel.initialize(nb_workers=10)


In [ ]:
# 初始化客户端
import os
HOST = os.environ['MODELONE_API_URL']
token = os.environ['MODELONE_API_TOKEN']
username = os.environ.get('MODELONE_USERNAME', 'admin')
init(host=HOST,username=username,token=token)

In [ ]:
# 生成一个加密秘钥
# from cryptography.fernet import Fernet
# key=Fernet.generate_key()
# print(key)
key = b'aViHLsGcYgmzMJrS98N2yRD3oTPMZf5JcZvKzr47f6E='

In [ ]:
# 定义一个数据集
dataset = Client(Dataset).one(name="coco")
if not dataset:
    dataset = Client(Dataset).add(name='coco', version='v2014', label='coco未标注数据集', describe='来自于2014年数据，未标注的coco数据集',icon='https://pic2.zhimg.com/80/v2-399df41d8562f8f09b98d288b97c8f8d_1440w.webp')

In [ ]:
# 上传数据集
features = json.dumps(json.load(open('vision/coco/data.json')),indent=4,ensure_ascii=False)
dataset = dataset.update(path='',features=features)
# 压缩
dataset.compress('coco.zip','vision/coco')
# 加密
dataset.encrypt('coco.zip',"coco.zip.crypt",key)
# 上传
dataset.upload('coco.zip.crypt',partition='20230201')

In [ ]:
# 下载数据集
os.remove('coco.zip.crypt')  if os.path.exists('coco.zip.crypt') else ''
os.remove('coco.zip')  if os.path.exists('coco.zip') else ''
shutil.rmtree('coco')  if os.path.exists('coco') else ''
dataset.download(partition='20230201')
dataset.decrypt("coco.zip.crypt",'coco.zip',key)
dataset.decompress('coco.zip','coco')

In [ ]:
import pandas
# 数据集加载
data = pandas.read_csv('coco/data.csv')
# 查看数据集的基本信息：
data.info()
# 查看数据集的统计描述：
data.describe()